In [19]:
import pandas as pd
import json
import re
from pathlib import Path

# Load the metadata parquet
import pandas as pd, json, re
import pyarrow.parquet as pq

df = pq.read_table("/home/bwilliams/mlx/week4/MLX8-W4-Multimodal-TransferLearning/.ben/data/reef_data/finaLmetadata.parquet").to_pandas()

# 2) Extract the model’s prediction
def extract_status(caption: str) -> str:
    m = re.search(r"```json\s*(\{.*?\})", caption, re.DOTALL)
    if not m:
        m = re.search(r"```json\s*(\[.*?\])", caption, re.DOTALL)
    if not m:
        return ""
    block = m.group(1)
    try:
        obj = json.loads(block)
        if isinstance(obj, list):
            obj = obj[0]
        return obj.get("health_status", "").upper()
    except json.JSONDecodeError:
        return ""

df["pred"] = df["caption"].apply(extract_status)

# 3) Derive the true label from the first letter of filename
map_true = {"h": "HEALTHY", "d": "DEGRADED", "r": "RESTORED"}
df["true"] = df["filename"].str[0].str.lower().map(map_true).fillna("UNKNOWN")

# 4) Counts
true_counts = df["true"].value_counts()
pred_counts = df["pred"].value_counts()

print("True label counts:\n", true_counts, "\n")
print("Predicted label counts:\n", pred_counts, "\n")

# 5) Per-class accuracy for H & D
for cls in ["HEALTHY", "DEGRADED"]:
    n_true = true_counts.get(cls, 0)
    n_corr = ((df["true"] == cls) & (df["pred"] == cls)).sum()
    acc   = (n_corr / n_true * 100) if n_true else 0.0
    print(f"{cls:>9}: true={n_true:4d}, correct={n_corr:4d}, acc={acc:5.1f}%")

# 6) Overall on H/D
hd = df[df["true"].isin(["HEALTHY", "DEGRADED"])]
overall_acc = (hd["true"] == hd["pred"]).mean() * 100
print(f"\nOverall HEALTHY/DEGRADED accuracy: {overall_acc:.1f}%")

# 7) Restored → predicted as Healthy
n_r = true_counts.get("RESTORED", 0)
n_rh = ((df["true"] == "RESTORED") & (df["pred"] == "HEALTHY")).sum()
pct_rh = (n_rh / n_r * 100) if n_r else 0.0
print(f"\nRESTORED: true={n_r}, predicted HEALTHY={n_rh} ({pct_rh:.1f}%)")


True label counts:
 true
HEALTHY     573
DEGRADED    211
Name: count, dtype: int64 

Predicted label counts:
 pred
DEGRADED    428
HEALTHY     341
             15
Name: count, dtype: int64 

  HEALTHY: true= 573, correct= 273, acc= 47.6%
 DEGRADED: true= 211, correct= 137, acc= 64.9%

Overall HEALTHY/DEGRADED accuracy: 52.3%

RESTORED: true=0, predicted HEALTHY=0 (0.0%)


In [ ]:

def print_metadata() -> None:
    """
    Load metadata.parquet and print the first 10 rows in full.
    """
    # adjust this path if you move the script
    metadata_path = Path("/home/bwilliams/mlx/week4/MLX8-W4-Multimodal-TransferLearning/.ben/data/reef_data/metadata.parquet")
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Metadata file not found at {metadata_path}")

    # read parquet (requires pandas + pyarrow)
    df = pd.read_parquet(metadata_path)
    # print all columns for the first 10 entries
    with pd.option_context("display.max_columns", None):
        print(df.head(10))

print_metadata()

             filename                                            caption split
0  d3_Right(1013).jpg  ```json\n{\n  "description": "The image shows ...  test
1  d3_Right(1110).jpg  ```json\n{\n  "description": "The image shows ...  test
2  d3_Right(1143).jpg  ```json\n{\n  "description": "The image shows ...  test
3  d3_Right(1176).jpg  ```json\n{\n  "description": "The image shows ...  test
4  d3_Right(1198).jpg  ```json\n{\n  "description": "The image shows ...  test
5   d3_Right(123).jpg  ```json\n[\n  {\n    "description": "The left ...  test
6  d3_Right(1273).jpg  ```json\n{\n  "description": "The image shows ...  test
7  d3_Right(1284).jpg  ```json\n{\n  "description": "The image shows ...  test
8  d3_Right(1316).jpg  ```json\n{\n  "description": "The image shows ...  test
9  d3_Right(1327).jpg  ```json\n{\n  "description": "The image shows ...  test


In [ ]:
import pyarrow.parquet as pq
import pandas as pd

# 1) Read the parquet (only the filename column)

table = pq.read_table("/home/bwilliams/mlx/week4/MLX8-W4-Multimodal-TransferLearning/.ben/data/reef_data/metadata.parquet", columns=["filename"])
df = table.to_pandas()

# 2) Total rows
print(f"Total rows in parquet: {len(df)}")

# 3) Count by first letter (lower-cased)
counts = df["filename"].str[0].str.lower().value_counts()

for letter in ["d", "h", "r"]:
    print(f"{letter.upper()}: {counts.get(letter, 0)}")


Total rows in parquet: 1741
D: 131
H: 0
R: 1610
